# 09B｜獨立測試資料切分

將 09A 裁切完成的 300 張圖片切為兩組互斥的資料：

- **校準集**：Real 50 張、Fake 50 張，共 100 張。用於選擇分類門檻。
- **正式測試集**：Real 100 張、Fake 100 張，共 200 張。用於評估選定門檻的效果。

門檻是一個需要從資料中決定的參數。若在同一批資料上既選門檻又報效能，得到的數字必然偏高，且無法反映實際部署時面對未知資料的表現。切成兩組是為了讓「選門檻」與「評估門檻」使用不同的樣本。

本 Notebook 只複製圖片，不移動、修改或刪除原始資料。

## 1. 匯入套件

In [1]:
from pathlib import Path
import hashlib
import json
import random
import shutil

import pandas as pd

## 2. 設定

`STOP_IF_OUTPUT_EXISTS = True`：輸出資料夾若已有內容即中止，避免不同批次的切分結果混在一起。

In [2]:
# YOLO 裁切後、尚未重新切分的 300 張圖片
SOURCE_ROOT = Path("./dataset_mobile_yolo/test")

# 新切分的輸出位置
OUTPUT_ROOT = Path("./dataset_mobile_split")

SEED = 42
CALIBRATION_PER_CLASS = 50
FORMAL_TEST_PER_CLASS = 100

CLASS_NAMES = ("real", "fake")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

# 為避免意外混入舊切分，輸出資料夾若已存在內容就停止
STOP_IF_OUTPUT_EXISTS = True

## 3. 工具函式

In [3]:
def list_images(folder):
    """遞迴取得資料夾內所有支援的圖片。"""
    return sorted(
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def sha256_file(path, chunk_size=1024 * 1024):
    """計算檔案內容雜湊，用於檢查重複圖片。"""
    digest = hashlib.sha256()
    with path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def ensure_empty_output(output_root):
    """若輸出位置已有檔案，停止執行以防止新舊資料混合。"""
    if output_root.exists() and any(output_root.rglob("*")):
        if STOP_IF_OUTPUT_EXISTS:
            raise FileExistsError(
                f"輸出資料夾已有內容：{output_root.resolve()}\n"
                "請先確認舊結果是否仍需要，再手動改名、移走或清空該資料夾。"
            )

## 4. 檢查原始資料

要求每類剛好 150 張。數量不符即中止，不自行忽略多出的圖片——靜默丟棄樣本會讓實際使用的資料與紀錄不一致。

In [4]:
required_per_class = CALIBRATION_PER_CLASS + FORMAL_TEST_PER_CLASS
source_images = {}

if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"找不到來源資料夾：{SOURCE_ROOT.resolve()}")

for class_name in CLASS_NAMES:
    class_dir = SOURCE_ROOT / class_name
    if not class_dir.exists():
        raise FileNotFoundError(f"找不到類別資料夾：{class_dir.resolve()}")

    images = list_images(class_dir)
    source_images[class_name] = images
    print(f"{class_name}: {len(images)} 張")

    if len(images) != required_per_class:
        raise ValueError(
            f"{class_name} 應有 {required_per_class} 張，目前是 {len(images)} 張。"
        )

print(f"總計：{sum(len(paths) for paths in source_images.values())} 張")

real: 150 張
fake: 150 張
總計：300 張


## 5. 檢查重複圖片

以 SHA-256 比對檔案內容。若存在完全相同的圖片，切分後可能同時落入校準集與測試集，使兩者不再互斥，門檻選擇的結果會直接洩漏到評估中。

檢查通過，300 張圖片內容互異。

In [5]:
hash_records = []

for class_name, paths in source_images.items():
    for path in paths:
        hash_records.append({
            "class_name": class_name,
            "source_path": str(path.resolve()),
            "sha256": sha256_file(path),
        })

hash_df = pd.DataFrame(hash_records)
duplicate_rows = hash_df[hash_df.duplicated("sha256", keep=False)].sort_values("sha256")

if not duplicate_rows.empty:
    display(duplicate_rows)
    raise ValueError("發現內容相同的重複圖片，請先確認並移除後再切分。")

print("重複圖片檢查通過。")

重複圖片檢查通過。


## 6. 固定 Seed 並建立切分

相同的原始檔案、Seed 與設定會產生完全相同的切分結果。

In [6]:
ensure_empty_output(OUTPUT_ROOT)
rng = random.Random(SEED)
split_plan = []

for class_name in CLASS_NAMES:
    shuffled = source_images[class_name].copy()
    rng.shuffle(shuffled)

    calibration_paths = shuffled[:CALIBRATION_PER_CLASS]
    formal_test_paths = shuffled[CALIBRATION_PER_CLASS:]

    for split_name, paths in (
        ("calibration", calibration_paths),
        ("test", formal_test_paths),
    ):
        for index, source_path in enumerate(paths, start=1):
            output_name = f"{class_name}_{split_name}_{index:03d}{source_path.suffix.lower()}"
            destination = OUTPUT_ROOT / split_name / class_name / output_name
            split_plan.append({
                "split": split_name,
                "class_name": class_name,
                "label": 0 if class_name == "real" else 1,
                "source_path": str(source_path.resolve()),
                "output_path": str(destination.resolve()),
                "output_name": output_name,
                "sha256": hash_df.loc[
                    hash_df["source_path"] == str(source_path.resolve()), "sha256"
                ].iloc[0],
            })

split_df = pd.DataFrame(split_plan)
display(pd.crosstab(split_df["split"], split_df["class_name"]))

class_name,fake,real
split,,
calibration,50,50
test,100,100


## 7. 複製圖片並保存切分紀錄

In [7]:
for row in split_df.itertuples(index=False):
    source_path = Path(row.source_path)
    output_path = Path(row.output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, output_path)

manifest_path = OUTPUT_ROOT / "split_manifest.csv"
config_path = OUTPUT_ROOT / "split_config.json"

split_df.to_csv(manifest_path, index=False, encoding="utf-8-sig")

split_config = {
    "source_root": str(SOURCE_ROOT.resolve()),
    "output_root": str(OUTPUT_ROOT.resolve()),
    "seed": SEED,
    "label_mapping": {"real": 0, "fake": 1},
    "calibration_per_class": CALIBRATION_PER_CLASS,
    "formal_test_per_class": FORMAL_TEST_PER_CLASS,
}

with config_path.open("w", encoding="utf-8") as file:
    json.dump(split_config, file, ensure_ascii=False, indent=2)

print(f"圖片已複製到：{OUTPUT_ROOT.resolve()}")
print(f"切分清單：{manifest_path.resolve()}")
print(f"切分設定：{config_path.resolve()}")

圖片已複製到：D:\資料\專題\PY\deepfake\新版專題研究\dataset_mobile_split
切分清單：D:\資料\專題\PY\deepfake\新版專題研究\dataset_mobile_split\split_manifest.csv
切分設定：D:\資料\專題\PY\deepfake\新版專題研究\dataset_mobile_split\split_config.json


## 8. 最終驗證

三項檢查：各組數量正確、複製後檔案內容的雜湊值與來源一致、校準集與測試集的雜湊值集合無交集。全數通過。

In [8]:
expected_counts = {
    ("calibration", "real"): CALIBRATION_PER_CLASS,
    ("calibration", "fake"): CALIBRATION_PER_CLASS,
    ("test", "real"): FORMAL_TEST_PER_CLASS,
    ("test", "fake"): FORMAL_TEST_PER_CLASS,
}

for (split_name, class_name), expected_count in expected_counts.items():
    folder = OUTPUT_ROOT / split_name / class_name
    actual_paths = list_images(folder)
    if len(actual_paths) != expected_count:
        raise RuntimeError(
            f"{split_name}/{class_name} 數量錯誤："
            f"預期 {expected_count}，實際 {len(actual_paths)}"
        )

    for output_path in actual_paths:
        recorded_hash = split_df.loc[
            split_df["output_path"] == str(output_path.resolve()), "sha256"
        ].iloc[0]
        if sha256_file(output_path) != recorded_hash:
            raise RuntimeError(f"複製後檔案內容不一致：{output_path}")

calibration_hashes = set(split_df.loc[split_df["split"] == "calibration", "sha256"])
test_hashes = set(split_df.loc[split_df["split"] == "test", "sha256"])

if calibration_hashes & test_hashes:
    raise RuntimeError("校準集與正式測試集出現重複圖片。")

print("最終驗證通過。")
print("校準集：100 張（Real 50、Fake 50）")
print("正式測試集：200 張（Real 100、Fake 100）")
print("兩個集合沒有重複圖片。")

最終驗證通過。
校準集：100 張（Real 50、Fake 50）
正式測試集：200 張（Real 100、Fake 100）
兩個集合沒有重複圖片。


## 輸出結果

```text
dataset_mobile_split/
├── calibration/   Real 50、Fake 50
└── test/          Real 100、Fake 100
```

`split_manifest.csv` 與 `split_config.json` 記錄完整的切分對應與設定。

09C 只使用 `calibration` 選擇門檻，選定後固定，再於 `test` 上評估。正式測試集不參與門檻選擇。